# 03 - メンバー間シナジー分析

どのメンバーの組み合わせが勝率が高いかを分析する。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from itertools import combinations
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df = pd.read_csv(DATA / 'player_stats.csv', parse_dates=['gameCreation'])
df['riotId'] = df['summonerName'] + '#' + df['tagLine']
df_m = df[df['riotId'].isin(MEMBERS)].copy()

In [ ]:
# 試合ごとに参加メンバーをグループ化
match_members = df_m.groupby('matchId').agg(
    members=('summonerName', lambda x: frozenset(x)),
    win=('win', 'first'),
    member_count=('summonerName', 'count'),
).reset_index()

print('参加人数別の試合数:')
print(match_members['member_count'].value_counts().sort_index())

In [ ]:
# デュオ（2人組み合わせ）別の勝率
all_members = sorted(df_m['summonerName'].unique())
duo_results = []

for a, b in combinations(all_members, 2):
    mask = match_members['members'].apply(lambda s: a in s and b in s)
    subset = match_members[mask]
    if len(subset) >= 3:
        duo_results.append({
            'duo': f'{a} + {b}',
            'games': len(subset),
            'wins': subset['win'].sum(),
            'winrate': round(subset['win'].mean() * 100, 1),
        })

df_duo = pd.DataFrame(duo_results).sort_values('winrate', ascending=False)
df_duo

In [ ]:
# デュオ勝率マトリクス（ヒートマップ）
matrix = pd.DataFrame(index=all_members, columns=all_members, dtype=float)

for a, b in combinations(all_members, 2):
    mask = match_members['members'].apply(lambda s: a in s and b in s)
    subset = match_members[mask]
    if len(subset) >= 3:
        wr = round(subset['win'].mean() * 100, 1)
        matrix.loc[a, b] = wr
        matrix.loc[b, a] = wr

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matrix.astype(float), annot=True, fmt='.0f', cmap='RdYlGn',
            center=50, vmin=30, vmax=70, ax=ax)
ax.set_title('デュオ勝率マトリクス (%) ≥3試合')
plt.tight_layout()
plt.show()

In [ ]:
# ロール組み合わせの勝率
role_pairs = df_m.groupby('matchId').apply(
    lambda g: pd.Series({
        'roles': ' / '.join(sorted(g['role'].values)),
        'win': g['win'].iloc[0],
    })
).reset_index()

role_wr = role_pairs.groupby('roles').agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
).round(2)
role_wr['winrate_pct'] = (role_wr['winrate'] * 100).round(1)
role_wr = role_wr[role_wr['games'] >= 3].sort_values('winrate', ascending=False)
role_wr